# 📊 Exploratory Data Analysis — APTOS 2019
Understand the dataset before training: class imbalance, image quality, preprocessing effects.

In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR  = '/content/data'
TRAIN_CSV = f'{DATA_DIR}/train.csv'
IMG_DIR   = f'{DATA_DIR}/train_images'

df = pd.read_csv(TRAIN_CSV)
print(f'Total samples: {len(df)}')
print(df['diagnosis'].value_counts().sort_index())

In [ ]:
# Class distribution pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

labels  = ['No DR','Mild','Moderate','Severe','Proliferative']
counts  = df['diagnosis'].value_counts().sort_index()
colors  = ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad']

ax1.pie(counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
ax1.set_title('Class Distribution (Pie)', fontsize=12)

ax2.bar(labels, counts, color=colors)
ax2.set_title('Class Distribution (Bar)', fontsize=12)
ax2.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Image resolution analysis
resolutions = []
for img_id in df['id_code'].sample(200, random_state=42):
    p = f'{IMG_DIR}/{img_id}.png'
    img = cv2.imread(p)
    if img is not None:
        resolutions.append(img.shape[:2])

resolutions = np.array(resolutions)
plt.figure(figsize=(6,4))
plt.scatter(resolutions[:,1], resolutions[:,0], alpha=0.4, color='steelblue', s=20)
plt.xlabel('Width (px)'); plt.ylabel('Height (px)')
plt.title('Image Resolution Distribution (200 samples)')
plt.tight_layout(); plt.show()
print(f'Height — min:{resolutions[:,0].min()} max:{resolutions[:,0].max()} mean:{resolutions[:,0].mean():.0f}')
print(f'Width  — min:{resolutions[:,1].min()} max:{resolutions[:,1].max()} mean:{resolutions[:,1].mean():.0f}')

In [ ]:
# Preprocessing effect visualisation
import sys; sys.path.insert(0,'/content/dr_project')
from utils.preprocess import apply_clahe, remove_black_border, subtract_local_mean

sample_id = df['id_code'].iloc[0]
img_bgr   = cv2.imread(f'{IMG_DIR}/{sample_id}.png')
img_rgb   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_small = cv2.resize(img_rgb, (512, 512))

stages = [
    ('Original',            img_small),
    ('Border Removed',      remove_black_border(img_small)),
    ('Local Mean Sub.',     subtract_local_mean(img_small)),
    ('CLAHE',               apply_clahe(img_small)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, img) in zip(axes, stages):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle('Preprocessing Pipeline', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Pixel intensity histograms per channel
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
channel_names = ['Red', 'Green', 'Blue']
colors_ch = ['red', 'green', 'blue']

for i, (name, color) in enumerate(zip(channel_names, colors_ch)):
    axes[i].hist(img_small[:,:,i].ravel(), bins=128, color=color, alpha=0.7)
    axes[i].set_title(f'{name} Channel')
    axes[i].set_xlabel('Pixel Intensity')
    axes[i].set_ylabel('Frequency')

plt.suptitle('RGB Channel Intensity Distributions', fontsize=11)
plt.tight_layout(); plt.show()